## Library

In [ ]:
import ccxt.async_support as ccxt
import pandas as pd
import numpy as np
import asyncio
import websockets
import json
import logging
import os
import dotenv
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
dotenv.load_dotenv()

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

# Utility / Exchange helper

In [ ]:
async def create_ccxt_exchange(api_key: str = API_KEY, secret: str = SECRET_KEY):
    exchange = ccxt.binance({
        "apiKey": api_key,
        "secret": secret,
        "enableRateLimit": True,
        "options": {"defaultType": "future"},  # change if using spot
    })
    await exchange.load_markets()
    return exchange

# Variables

In [ ]:
SYBLOL = "BTC/USDT"
INTERVAL = "1m"

In [ ]:
while True:
    try:
        # Fetch Trade For Last Hours
        since = int((time.time() - 60 * 60) * 1000)
        trade = exchange.fetch_trades(symbol=SYBLOL, since=since, limit=1000)
        df_trades = pd.DataFrame(trade)
        
        # Process Trades DataFrame
        df_trades['timestamp'] = pd.to_datetime(df_trades['timestamp'], unit='ms')
        df_trades['volume'] = df_trades['amount'] * df_trades['price']
        df_trades['value'] = df_trades['side'].apply(lambda x: 1 if x == 'buy' else -1) * df_trades['volume']
        df_trades['volume']
        
        
        # Calculate inflow (Buy Value) and outflow (Sell Value)
        inflow = df_trades[df_trades['side'] == 'buy']['value'].sum()
        outflow = df_trades[df_trades['side'] == 'sell']['value'].sum()
        
        # Calculate net flow
        net_flow = inflow - outflow
        
        # Print results
        Print(f"Inflow: {inflow}, Outflow: {outflow}, Net Flow: {net_flow}")
        
        time.sleep(3600 - (time.time() % 3600))  # Wait for 1 minute before next fetch
    except Exception as e:
        logger.error(f"Error fetching trades: {e}")
        await asyncio.sleep(10)  # Wait before retrying

# Runner

In [ ]:
async def run():
    exchange = await create_ccxt_exchange()
    try:
        await main_loop(exchange, SYMBOL)
    finally:
        await exchange.close()

if __name__ == "__main__":
    try:
        asyncio.run(run())
    except KeyboardInterrupt:
        logger.info("Stopped by user")